# MSI5001 — L3 Feature Engineering Demonstration
**Reproducible, label-blind generation of V/N/S features and the L3 master feature matrix**

This notebook demonstrates the merged L3 implementation. It reads **only** RAW `text` and P1 `processed_text`; ground-truth sentiment and confidence columns are never loaded. It produces four CSVs and a JSON SHA256 manifest in Google Drive, shows summary tables, and validates against the audited corpus counts.

**Expected:** 14,639 tweets, 10 engineered features, master shape (14639, 11). No scaling, clustering, tuning or label-based feature selection.

Run from top to bottom in **Google Colab**. This notebook is intentionally committed to GitHub with project authorization; generated datasets remain in Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Clone the approved main branch and install dependencies
This demonstration uses the **merged main branch** rather than a development branch. A fresh Colab runtime is recommended.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO = Path("/content/sentiment-clustering-ml")
if REPO.exists():
    assert (REPO / ".git").is_dir(), f"Existing directory is not a Git repository: {REPO}"
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "switch", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main",
        "https://github.com/frankie-chew-nus/sentiment-clustering-ml.git", str(REPO)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-dev.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "vaderSentiment==3.3.2"], check=True)
print("Git revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
sys.path.insert(0, str(REPO))

## 2. Locate and verify RAW and P1 inputs
The P1 path follows the project's `0.Processed-files` convention. Change it only if your validated P1 CSV is stored elsewhere. SHA256 checks prevent accidental use of a different version.

In [ ]:
from pathlib import Path
from feature_engineering import (
    load_raw_text_only, load_p1_text_only, extract_vader_frame,
    extract_n_frame, extract_s_frame, raw_csv_fingerprint,
)
from master_feature_matrix import assemble_csvs, MASTER_COLUMNS, sha256_file
import pandas as pd
import numpy as np
import json
from IPython.display import display

BASE = Path("/content/drive/MyDrive/MSI5001 project/EDA")
RAW_PATH = BASE / "Tweets-redacted.csv"
P1_PATH = BASE / "0.Processed-files" / "Tweets-processed-p1.csv"
OUT = BASE / "L3_Feature_Engineering"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_SHA = "a41aeefcff9740c090153f608691d345c84f59084e40f03169c2f667df66578e"
EXPECTED_P1_SHA = "fdeab9faa57b7ffb7ac33335aebafc5f153849d94647e0118d02e809a86c632e"
for label,path,expected in [("RAW",RAW_PATH,EXPECTED_RAW_SHA),("P1",P1_PATH,EXPECTED_P1_SHA)]:
    assert path.is_file(), f"Missing {label}: {path}"
    actual = sha256_file(path)
    print(label, path.name, actual)
    assert actual == expected, f"{label} hash differs from audited source"
print("Both source fingerprints match the audited files.")

## 3. Generate V, N and S family CSVs
VADER and structural expression use RAW text; negation/intensity uses validated P1. These are **unscaled** features. The `row_id` is alignment metadata, not a clustering feature.

In [ ]:
raw_texts = load_raw_text_only(RAW_PATH)
p1_texts = load_p1_text_only(P1_PATH)
assert len(raw_texts) == len(p1_texts) == 14639

v = extract_vader_frame(raw_texts)
n = extract_n_frame(p1_texts)
s = extract_s_frame(raw_texts)

V_PATH = OUT / "Tweets-L3-vader-features.csv"
N_PATH = OUT / "Tweets-L3-negation-intensity-features.csv"
S_PATH = OUT / "Tweets-L3-structural-features.csv"
for frame,path in [(v,V_PATH),(n,N_PATH),(s,S_PATH)]:
    frame.to_csv(path,index=False)
    print(path.name, frame.shape)
display(v.head(3))
display(n.head(3))
display(s.head(3))

## 4. Verify extraction parity with the label-blind audits
The checks below are pre-agreed descriptive counts. They are **not** sentiment labels or clustering-performance metrics.

In [ ]:
observed = {
    "tweets": len(v),
    "vader_compound_zero": int((v.V_compound == 0).sum()),
    "vader_pos_and_neg_zero": int(((v.V_pos == 0) & (v.V_neg == 0)).sum()),
    "vader_compound_unique": int(v.V_compound.nunique()),
    "negation_present": int((n.N_negation_count > 0).sum()),
    "intensifier_original_present": int((n.N_intensifier_original_count > 0).sum()),
    "intensifier_conservative_present": int((n.N_intensifier_conservative_count > 0).sum()),
    "exclamation_present": int((s.S_exclamation_count > 0).sum()),
    "emoji_present": int((s.S_emoji_count > 0).sum()),
}
expected = {
    "tweets":14639, "vader_compound_zero":2323,
    "vader_pos_and_neg_zero":2269, "vader_compound_unique":2915,
    "negation_present":4916, "intensifier_original_present":1823,
    "intensifier_conservative_present":693,
    "exclamation_present":3226, "emoji_present":492,
}
checks = pd.DataFrame([{"metric":k,"expected":expected[k],"actual":observed[k],
                        "status":"PASS" if expected[k]==observed[k] else "FAIL"} for k in expected])
display(checks)
assert observed == expected, "Audit parity mismatch: investigate before assembly"

## 5. Assemble the master feature matrix and SHA256 manifest
The assembler validates strict schemas, numeric ranges, row order and unchanged values. It rejects any unexpected columns, including sentiment labels/confidence.

In [ ]:
MASTER_PATH = OUT / "Tweets-L3-master-features.csv"
master, manifest_path = assemble_csvs(
    vader_path=V_PATH,
    negation_path=N_PATH,
    structural_path=S_PATH,
    output_path=MASTER_PATH,
    source_sha256={"RAW":EXPECTED_RAW_SHA, "P1":EXPECTED_P1_SHA},
)
assert master.shape == (14639,11)
assert master.columns.tolist() == list(MASTER_COLUMNS)
print("Master:",MASTER_PATH)
print("Manifest:",manifest_path)
print("Master shape:",master.shape)
display(master.head(10))

## 6. Display presentation-ready evidence
Show the full summary statistics and feature prevalence. **Nonzero prevalence is not a sentiment class distribution**; negative compound values count as nonzero.

In [ ]:
feature_cols = [c for c in master.columns if c!="row_id"]
summary = master[feature_cols].describe().T.round(6)
prevalence = pd.DataFrame({
    "feature":feature_cols,
    "nonzero_tweets":[int((master[c]!=0).sum()) for c in feature_cols],
    "nonzero_pct":[round(100*(master[c]!=0).mean(),2) for c in feature_cols],
})
print("Summary statistics:")
display(summary)
print("Feature prevalence:")
display(prevalence)

## 7. Final integrity and provenance report
A manifest records **input/output file hashes**. RAW/P1 hashes are reference values from the validated source files; this notebook also verifies those files directly before extraction.

In [ ]:
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
checks = {
    "14,639 rows": len(master)==14639,
    "11 columns": len(master.columns)==11,
    "row_id 0..14638": master.row_id.tolist()==list(range(14639)),
    "no duplicate row_id": not master.row_id.duplicated().any(),
    "no missing feature values": not master[feature_cols].isna().any().any(),
    "finite features": bool(np.isfinite(master[feature_cols].to_numpy(dtype=float)).all()),
    "no sentiment or confidence columns": not any("sentiment" in c.lower() or "confidence" in c.lower() for c in master.columns),
    "master hash matches manifest": sha256_file(MASTER_PATH)==manifest["output_sha256"],
    "input hashes match manifest": all(
        sha256_file(path)==manifest["input_files"][family]["sha256"]
        for family,path in [("V",V_PATH),("N",N_PATH),("S",S_PATH)]
    ),
    "unscaled": manifest["scaled"] is False,
    "labels not loaded": manifest["sentiment_labels_loaded"] is False,
}
display(pd.DataFrame([{"check":k,"result":"PASS" if v else "FAIL"} for k,v in checks.items()]))
assert all(checks.values()), "Master integrity check failed"
print("Master SHA256:",manifest["output_sha256"])
print("Output directory:",OUT)
print("L3 FEATURE GENERATION DEMO: PASSED")
print("Note: This verifies extraction/assembly, not sentiment-clustering effectiveness.")